<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Machine Learning Fundamentals** · Week 1 · Your First Models with scikit-learn · Lesson 6 of 24

# Week 1 Project: Predict Rents in Kigali

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals).

**Learning objective:** Run a complete regression project from raw data to a short report: prepare the features, compare three models against a baseline, and explain the winner.

## The brief

A property website in Kigali wants to show visitors a **fair rent estimate** for any home. You are the data scientist. Build the model, test it honestly and write a one-page report for the manager.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 600
district = rng.choice(["Gasabo", "Kicukiro", "Nyarugenge"], n, p=[0.45, 0.35, 0.2])
bedrooms = rng.integers(1, 5, n)
size_m2 = (bedrooms * 28 + rng.normal(20, 12, n)).clip(25).round()
km_to_centre = rng.uniform(0.5, 15, n).round(1)
furnished = rng.random(n) < 0.3
base = {"Gasabo": 120_000, "Kicukiro": 90_000, "Nyarugenge": 100_000}
rent = (np.array([base[d] for d in district]) + 2_600 * size_m2 - 6_000 * km_to_centre
        + np.where(furnished, 60_000, 0) + rng.normal(0, 25_000, n)).round(-3)
homes = pd.DataFrame({"district": district, "bedrooms": bedrooms, "size_m2": size_m2,
                      "km_to_centre": km_to_centre, "furnished": furnished, "rent_rwf": rent})
print(homes.shape)
homes.head()

## Step 1 — Explore (15 minutes)

- `homes.describe()` and `homes["district"].value_counts()`.
- Average rent per district (`groupby`).
- A scatter plot of `size_m2` against `rent_rwf`, and one of `km_to_centre` against rent.

Write two sentences on what you see.

## Step 2 — Prepare

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = pd.get_dummies(homes.drop(columns="rent_rwf"), columns=["district"], dtype=int)
X["furnished"] = X["furnished"].astype(int)
y = homes["rent_rwf"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

## Step 3 — Compare models

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, r2_score

models = {
    "Baseline (mean)": DummyRegressor(),
    "Linear regression": LinearRegression(),
    "k-NN (k=10)": KNeighborsRegressor(n_neighbors=10),
}
results = []
for name, m in models.items():
    m.fit(X_train_s, y_train)
    pred = m.predict(X_test_s)
    results.append({"model": name,
                    "MAE": round(mean_absolute_error(y_test, pred)),
                    "R2": round(r2_score(y_test, pred), 3)})
print(pd.DataFrame(results))

Linear regression should win clearly (MAE about 20,000 RWF), k-NN comes second and the baseline is far behind. That makes sense: the rents were built from a straight-line rule, which is exactly what linear regression can learn.

## Step 4 — Explain the winner

- Print the coefficients of a linear model trained on the **unscaled** features (they are easier to read in RWF).
- Which three features matter most? Explain them in plain words.
- Plot predicted vs real rent for the test homes.

## Step 5 — Use it

Estimate the rent for three example homes of your choice (one per district). Present the answers rounded to the nearest 5,000 RWF — false precision looks unprofessional.

## Step 6 — Report (one page)

1. **Goal** — one sentence.
2. **Data** — 600 simulated homes; the features used.
3. **Results** — a small table of the three models with MAE and R².
4. **What drives rent** — the top three features, in RWF.
5. **Limits** — at least two (for example: simulated data; no information on building quality, water or security; rents change every year).

## Marking guide (self-check)

| Area | Excellent work |
|---|---|
| Preparation | All features numeric, split before scaling |
| Comparison | Baseline included, same test set for every model |
| Explanation | Coefficients explained in plain RWF terms |
| Honesty | Limits stated clearly |
| Communication | A manager understands it in five minutes |

**Takeaway:** A real project is a story: explore, prepare, compare against a baseline, explain the winner and state its limits.

## Try it yourself

Finish Steps 1–6 in one notebook. Then challenge yourself: remove the `district` columns and retrain. How much worse does the model get? What does that tell the property website?

---
Finished? Go back to [Machine Learning Fundamentals on WAMUGII](https://wamugii.kento.rw/courses/machine-learning-fundamentals), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*